# AgentFence.AI — Module 2: User Intent Processor

## Objective

Convert each user task into structured information describing what the user intends to do. Later, AgentFence.AI can compare this intent with the access or actions requested by an application.

## Input

This notebook reads `synthetic_scenarios.csv`, created by Module 1.

## Processing Approach

This first version uses transparent rules and does not require a paid artificial intelligence (AI) service. It reads only `scenario_id` and `user_task_text`.

It does not use `app_request_text` or `expected_decision`. This keeps the interpretation of the user’s intent separate from the application’s request and the expected result.

If a task does not match a known pattern, the notebook marks it `needs_review` instead of guessing.

## Output Files

- `normalized_intents.csv` — structured user intent records.
- `intent_processing_summary.json` — counts of matched and unrecognized tasks.

Module 4 will compare these intent records with the normalized application requests from Module 3.

In [1]:
from pathlib import Path
import csv
import json

input_root = Path("/kaggle/input")

def find_input_file(filename):
    matches = sorted(input_root.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach Notebook 1's output "
            "as an input, then run this notebook again."
        )
    return matches[0]

scenarios_path = find_input_file("synthetic_scenarios.csv")

with scenarios_path.open(newline="", encoding="utf-8") as file:
    scenarios = list(csv.DictReader(file))


def normalize_intent(task_text):
    text = task_text.casefold()

    if "upload" in text and ("résumé" in text or "resume" in text):
        return {
            "intent_status": "matched",
            "purpose": "Submit one résumé to a job application website.",
            "intended_action": "upload_selected_file",
            "allowed_scope": "one_selected_file",
            "allowed_recipient": "job_application_website",
            "duration_limit": "one_time",
            "source_change_limit": "no_source_changes",
            "outbound_transfer_limit": "one_file_to_job_application_website",
            "parser_basis": "The task says to upload one résumé.",
        }

    if "share" in text and "document" in text:
        recipient = (
            "one_accountant" if "accountant" in text
            else "one_named_colleague"
        )
        return {
            "intent_status": "matched",
            "purpose": "Share one document with one named recipient for review.",
            "intended_action": "share_selected_file",
            "allowed_scope": "one_selected_file",
            "allowed_recipient": recipient,
            "duration_limit": "purpose_limited_review",
            "source_change_limit": "no_source_changes",
            "outbound_transfer_limit": "one_file_to_stated_recipient",
            "parser_basis": "The task says to share one document with a named person.",
        }

    if "schedule" in text or "meeting" in text:
        recipient = (
            "two_selected_colleagues"
            if "two selected colleagues" in text
            else "selected_attendees"
        )
        return {
            "intent_status": "matched",
            "purpose": "Schedule one meeting with selected attendees.",
            "intended_action": "create_one_calendar_event",
            "allowed_scope": "one_event_on_selected_calendar",
            "allowed_recipient": recipient,
            "duration_limit": "one_task",
            "source_change_limit": "create_one_event_only",
            "outbound_transfer_limit": "calendar_invitation_to_selected_attendees",
            "parser_basis": "The task says to schedule a meeting.",
        }

    if "summarize" in text and "page" in text:
        return {
            "intent_status": "matched",
            "purpose": "Summarize the currently open page for the user.",
            "intended_action": "read_and_summarize",
            "allowed_scope": "current_browser_tab",
            "allowed_recipient": "the_user",
            "duration_limit": "while_in_use",
            "source_change_limit": "no_source_changes",
            "outbound_transfer_limit": "none",
            "parser_basis": "The task says to summarize the page currently open.",
        }

    if "summarize" in text and "document" in text:
        return {
            "intent_status": "matched",
            "purpose": "Summarize one selected document for the user.",
            "intended_action": "read_and_summarize",
            "allowed_scope": "one_selected_file",
            "allowed_recipient": "the_user",
            "duration_limit": "one_time",
            "source_change_limit": "no_source_changes",
            "outbound_transfer_limit": "none",
            "parser_basis": "The task says to summarize one selected document.",
        }

    return {
        "intent_status": "needs_review",
        "purpose": "Could not confidently identify the user’s purpose.",
        "intended_action": "unknown",
        "allowed_scope": "unknown",
        "allowed_recipient": "unknown",
        "duration_limit": "unknown",
        "source_change_limit": "unknown",
        "outbound_transfer_limit": "unknown",
        "parser_basis": "No supported task pattern matched.",
    }


normalized_intents = []

for scenario in scenarios:
    intent = normalize_intent(scenario["user_task_text"])
    normalized_intents.append({
        "scenario_id": scenario["scenario_id"],
        "raw_intent_text": scenario["user_task_text"],
        **intent,
    })

output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

intents_path = output_dir / "normalized_intents.csv"
fieldnames = list(normalized_intents[0].keys())

with intents_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(normalized_intents)

status_counts = {}
for intent in normalized_intents:
    status = intent["intent_status"]
    status_counts[status] = status_counts.get(status, 0) + 1

summary = {
    "notebook": "AgentFence.AI Module 2: User Intent Processor",
    "input_file": str(scenarios_path),
    "scenario_count": len(normalized_intents),
    "intent_status_counts": status_counts,
    "output_file": str(intents_path),
}

summary_path = output_dir / "intent_processing_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(f"Processed {len(normalized_intents)} user task descriptions.")
print(f"Intent records: {intents_path}")
print(f"Processing summary: {summary_path}")
print(f"Status counts: {status_counts}")

Processed 15 user task descriptions.
Intent records: /kaggle/working/normalized_intents.csv
Processing summary: /kaggle/working/intent_processing_summary.json
Status counts: {'matched': 15}
